# Background subtraction

Example workflow for background removal in iDPC/HAADF tilt series. 
<br>Includes: mask generation from ADF, quick background subtraction via averaging, two inpainting background subtraction options (CV2 Telea and skimage biharmonic), and intensity normalisation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

import scipy.ndimage as ndi

from skimage import filters, morphology
from skimage.transform import resize
from skimage.restoration import inpaint, inpaint_biharmonic

from mpl_toolkits.mplot3d import Axes3D

import hyperspy.api as hs

%matplotlib widget

In [ ]:
def smooth_binary_mask(mask, sigma=2, threshold=0.5):
    blurred = ndi.gaussian_filter(mask.astype(float), sigma=sigma)
    smoothed_mask = blurred > threshold
    return smoothed_mask

def remove_background(projections, optimal_thresholds):
    filt = []
    results = []
    mask = []
    for i in range(projections.shape[0]):
        data = projections[i]-(optimal_thresholds[i]/projections.shape[2])
        results.append(data)
        gauss = ndi.gaussian_filter(projections[i], sigma=2)
        filt.append(gauss)
        thresh = filters.threshold_otsu(gauss, nbins = 3)
        binary = gauss < thresh
        binaryRefine = morphology.remove_small_objects(binary, min_size=90)
        smoothed_mask = smooth_binary_mask(~binaryRefine, sigma=20, threshold=0.02)
        mask.append(smoothed_mask)
        
    projfilt = np.stack(filt, axis=0)
    results = np.stack(results, axis=0)    
    mask = np.stack(mask, axis=0)
    results[~mask] = 0
    results[np.where(results<0)] =0
    return mask, results

def remove_average_background(projections, sigma=1):
    results = []
    masklist = []
    dist_mask = []
    meanlist = []
    for i in range(projections.shape[0]):
        gauss = ndi.gaussian_filter(projections[i], sigma=sigma)
        thresh = filters.threshold_otsu(gauss)
        binary = gauss < thresh
        binaryRefine = morphology.remove_small_objects(binary, min_size=90)
        smoothed_mask = smooth_binary_mask(~binaryRefine, sigma=10, threshold=0.02)
        masklist.append(smoothed_mask)
        distance_map = ndi.distance_transform_edt(~smoothed_mask)
        distmask = (distance_map >= 0.01) & (distance_map <= 50)
        mean = np.mean(projections[i][distmask])
        data = projections[i]-mean
        results.append(data)
        dist_mask.append(distmask)
        meanlist.append(mean)
        print(f'Tilt image {i} Completed!')
    results = np.stack(results, axis=0)    
    mask = np.stack(masklist, axis=0)
    results[~mask] = 0
    results[np.where(results<0)] =0
    dist_mask = np.stack(dist_mask, axis=0)
    return mask, dist_mask, results, meanlist

def remove_average_background_local(projections, sigma=10, block_size=299, smooth=True, smoothsigma=10, smooththres=0.02):
    projections = np.nan_to_num(projections)
    results = []
    dist_mask = []
    meanlist = []
    masklist =[]
    for i in range(projections.shape[0]):
        gauss = ndi.gaussian_filter(projections[i], sigma=sigma)
        thresh = filters.threshold_local(gauss,block_size=block_size)
        binary = gauss > thresh
        if smooth==True:
            binary = smooth_binary_mask(binary, sigma=smoothsigma, threshold=smooththres)
        binaryRefine = morphology.remove_small_objects(binary, min_size=90)
        masklist.append(binaryRefine)
        distance_map = ndi.distance_transform_edt(~binaryRefine)
        distmask = (distance_map >= 0.01) & (distance_map <= 50)
        mean = np.mean(projections[i][distmask])
        data = projections[i]-mean
        results.append(data)
        dist_mask.append(distmask)
        meanlist.append(mean)
        print(f'Tilt image {i} Completed!')

    results = np.stack(results, axis=0)    
    mask = np.stack(masklist, axis=0)
    #results = results - minVal # minus the negative to set to 0
    results[np.where(results<0)] =0
    results[~mask] = 0
    return masklist, dist_mask, results, meanlist

def remove_average_background_mask(projections, maskdata, distmasks, maskhspy = False):
    results = []
    dist_mask = distmasks
    meanlist = []
    masklist = maskdata
    if maskhspy == False:
        masks = np.stack(maskdata, axis=0)
    else:
        masks=maskdata
    for i in range(projections.shape[0]):
        distmask = dist_mask[i]
        mean = np.mean(projections[i][distmask])
        data = projections[i]-mean
        results.append(data)
        meanlist.append(mean)
        print(f'Tilt image {i} Completed!')
    
    results = np.stack(results, axis=0)    
    #results[np.where(results<0)] = 0
    results[~masks] = 0
    return masklist, dist_mask, results, meanlist

def remove_average_background_threshold(projections, sigma=1, thresh_variation = 10000):
    results = []
    masklist = []
    dist_mask = []
    meanlist = []
    
    for i in range(projections.shape[0]):
        gauss = ndi.gaussian_filter(projections[i], sigma=sigma)
        thresh = filters.threshold_otsu(gauss)+ thresh_variation
        binary = gauss < thresh
        binaryRefine = morphology.remove_small_objects(binary, min_size=90)
        smoothed_mask = smooth_binary_mask(~binaryRefine, sigma=10, threshold=0.01)
        masklist.append(smoothed_mask)
        distance_map = ndi.distance_transform_edt(~smoothed_mask)
        distmask = (distance_map >= 0.01) & (distance_map <= 50)
        mean = np.mean(projections[i][distmask])
        data = projections[i]-mean
        results.append(data)
        dist_mask.append(distmask)
        meanlist.append(mean)
        print(f'Tilt image {i} Completed!')
        
    results = np.stack(results, axis=0)    
    mask = np.stack(masklist, axis=0)
    results[~mask] = 0
    results[np.where(results<0)] =0
    dist_mask = np.stack(dist_mask, axis=0)
    return mask, dist_mask, results, meanlist

def distmask_calc(mask, mindist=0.01, maxdist=50):
    distance_map = ndi.distance_transform_edt(~mask)
    distmask = (distance_map >= mindist) & (distance_map <= maxdist)
    return distmask

In [ ]:
def intgrad2d(gradient: np.ndarray, sampling: tuple[float, float] = None):
    """
    Perform Fourier-space integration of gradient. Taken from the beta version of abTEM

    Parameters
    ----------
    gradient : two np.ndarrays
        The x- and y-components of the gradient.
    sampling : two float
        Lateral sampling of the gradients. Default is 1.0.

    Returns
    -------
    np.ndarray
        Integrated center of mass measurement
    """
    gx, gy = gradient
    (nx, ny) = gx.shape
    ikx = np.fft.fftfreq(nx, d=sampling[0])
    iky = np.fft.fftfreq(ny, d=sampling[1])
    grid_ikx, grid_iky = np.meshgrid(ikx, iky, indexing='ij')
    k = grid_ikx ** 2 + grid_iky ** 2
    k[k == 0] = 1e-12
    That = (np.fft.fft2(gx) * grid_ikx + np.fft.fft2(gy) * grid_iky) / (2j * np.pi * k)
    T = np.real(np.fft.ifft2(That))
    T -= T.min()
    return T

def DPC_to_iDPC(DPC1, DPC2, DPC3, DPC4):
    """
    Perform transformation from DPC to iDPC. Adapated for Hyperspy.
    
    Parameters
    ----------
    Input of DPC segments is dependent on your detector rotation.
    For Cardiff TF Spectra:
    DFS(0,4)=DPC1
    DFS(1,5)=DPC2
    DFS(3,7)=DPC3
    DFS(5,8)=DPC4
    
    Returns
    -------
    np.ndarray
        Integrated DPC measurement
    """
    signal_01 = np.mean(np.array([ DPC1, DPC2 ]), axis=0 )
    signal_23 = np.mean(np.array([ DPC3, DPC4 ]), axis=0 )
    signal_03 = np.mean(np.array([ DPC1, DPC4 ]), axis=0 )
    signal_21 = np.mean(np.array([ DPC3, DPC2 ]), axis=0 )
    differential_signal_y = signal_01 - signal_23
    differential_signal_x = signal_03 - signal_21
    grad= differential_signal_y,differential_signal_x

    idpc = intgrad2d(grad,[1,1])
    return idpc


In [ ]:
def normalise_to_reference_with_background(image, reference_image, mask):
    """
    Normalise an image to match the total intensity of a reference image while ensuring:
    - Background areas are first set to the minimum signal value.
    - The background is then shifted to zero, avoiding negativity.

    Parameters:
    - image: 2D numpy array of the image.
    - reference_image: 2D numpy array of the reference image.

    Returns:
    - Normalised image with the same total intensity as the reference.
    """
    # inverts mask to be of signal, and ensures it is set as 0 and 1.
    mask[mask>0.01] = 1
    signal_mask = ~mask

    # ensure positive:
    min_signal_value = np.min(image[signal_mask]) if np.any(signal_mask) else 0
    
    image_corrected = np.copy(image)
    image_corrected = image_corrected - min_signal_value #(set min to 0)
    image_corrected[~signal_mask] = 0 # set background also to 0

    print(image_corrected.shape)

    # Compute total intensity of the signal region
    current_sum = np.sum(image_corrected[signal_mask])
    print(current_sum)
    reference_sum = np.sum(reference_image[signal_mask]) # note you might want to adjust this as an input beforehand

    # Avoid divide by zero
    if current_sum == 0:
        return image_shifted

    scale_factor = reference_sum / current_sum
    print(scale_factor, 'image', i)
    # Scale only the signal pixels
    normalised_image = np.zeros_like(image_corrected)
    normalised_image[signal_mask] = image_corrected[signal_mask] * np.round(scale_factor,decimals=5)

    return image_corrected, normalised_image
    
def normalise_to_reference_with_sum(image, refsum, mask):
    """
    Normalise an image to match the total intensity of a reference image
    Background areas are first set to the minimum signal value, then shifted to be the new 0.
    """
    # inverts mask to be of signal, and ensures it is set as 0 and 1.
    mask[mask>0.01] = 1
    signal_mask = ~mask

    # ensure positive:
    min_signal_value = np.min(image[signal_mask]) if np.any(signal_mask) else 0
    
    image_corrected = np.copy(image)
    image_corrected = image_corrected - min_signal_value #(set min to 0)
    image_corrected[~signal_mask] = 0 # set background also to 0

    print(image_corrected.shape)

    # Compute total intensity of the signal region
    current_sum = np.sum(image_corrected[signal_mask])
    print(current_sum)
    reference_sum = refsum # note you might want to adjust this as an input beforehand

    # Avoid division by zero
    if current_sum == 0:
        return image_corrected

    # Compute scaling factor
    scale_factor = reference_sum / current_sum
    print(scale_factor, 'image', i)
    # Scale only the signal pixels
    normalised_image = np.zeros_like(image_corrected)
    normalised_image[signal_mask] = image_corrected[signal_mask] * (scale_factor)

    return image_corrected, normalised_image

# Loading tilt series

In [ ]:
# Load hspy file with tilt series. See zenodo - the tilt series are large (~2GB) so if need be, bin them or load your own test data!
image = hs.load(r"iDPC_FSP_031224_d4.hspy")
imagehaadf = hs.load(r"HAADFdenoise_5-23d4CeO2.hspy")

In [ ]:
idpcsplit = image.split()
haadfsplit = imagehaadf.split()
sort_tilt1 = sorted(idpcsplit, key=lambda idpcsplit: idpcsplit.metadata['Acquisition_instrument']['TEM']['Stage'].tilt_alpha)
sort_tilt2 = sorted(haadfsplit, key=lambda haadfsplit: haadfsplit.metadata['Acquisition_instrument']['TEM']['Stage'].tilt_alpha)

In [ ]:
i = 26
print(sort_tilt1[i].metadata['General']['original_filename'])
print(sort_tilt2[i].metadata['General']['original_filename'])
# to remove duplicate tilt angles use del sort_tilt1[i] etc

In [ ]:
stackhaadf = hs.stack(sort_tilt2)
stackidpc = hs.stack(sort_tilt1)
# can save these here with stackhaadf.save etc

if you know what indices to remove in advance, can use the below cell. Be sure to feed the same indices to ADF and DPC!

In [ ]:
#indices_to_remove = [3,5,7] # etc etc  
#sort_tiltrm = [item for i, item in enumerate(sort_tilt) if i not in indices_to_remove]

In [ ]:
haadfsplit.plot()

In [ ]:
idpcsplit.plot()

## Mask maker from ADF image

In [ ]:
projections = imagehaadf.data
masklist=[]

for i in range(projections.shape[0]):
    gauss = ndi.gaussian_filter(projections[i], sigma=1)
    binary = gauss > 0
    binaryRefine = morphology.remove_small_objects(binary, min_size=100000)
    masklist.append(binaryRefine)

binaryhs = hs.signals.Signal2D(masklist, metadata = imagehaadf.metadata.as_dictionary())
distmasks = []
for i in range(binaryhs.data.shape[0]):
    distmask = distmask_calc(binaryhs.data[i])
    distmasks.append(distmask)

binaryhs.plot()

## Option 1: Averaging background outside of mask

Good for a very quick, but less accurate removal. only use for initial testing or approximation!

In [ ]:
results = []
masklist = []
dist_mask = []
meanlist = []

In [ ]:
haadfsplit.plot()

Change `i` in the below block to the projection. Use the otsu filter as a baseline and manually adjust based on the plotted result to get the best mask. 

In [ ]:
i=0 
gauss = ndi.gaussian_filter(haadfsplit.data[i], sigma=10)
thresh = filters.threshold_otsu(gauss)-4000 # can use local or otsu, and add manual adjustments after.
binary = gauss < thresh
smoothed_mask = smooth_binary_mask(~binary, sigma=10, threshold=0.02) 
binaryRefine = morphology.remove_small_objects(smoothed_mask, min_size=90)

masklist.append(smoothed_mask)
distance_map = ndi.distance_transform_edt(~smoothed_mask)
distmask = (distance_map >= 0.01) & (distance_map <= 50)
mean = np.mean(imagehaadf.data[i][distmask])
data = imagehaadf.data[i]-mean
data[~binaryRefine] = 0
data[np.where(data<0)] = 0

In [ ]:
projections_average_hs = hs.signals.Signal2D(data, metadata = imagehaadf.metadata.as_dictionary())
projections_average_hs.plot()

Once happy with the above plot, run the below cell to append the result for that image, then adjust `i` to start the next image.

In [ ]:
results.append(data)
dist_mask.append(distmask)
meanlist.append(mean)

In [ ]:
projections_average_hs = hs.signals.Signal2D((results), metadata = imagehaadf.metadata.as_dictionary())

Test plot using the below cell to get threshold value for auto function.

In [ ]:
image = imagehaadf.data[25]
gauss = ndi.gaussian_filter(image, sigma=10)
thresh = filters.threshold_local(gauss, block_size=999)
binary = gauss > thresh
smoothed_mask = smooth_binary_mask(binary, sigma=10, threshold=0.02)
binaryRefine = morphology.remove_small_objects(smoothed_mask, min_size=90)

# check plot results
fig, axes = plt.subplots(ncols=3, figsize=(8, 2.5))
ax = axes.ravel()
ax[0] = plt.subplot(1, 3, 1)
ax[1] = plt.subplot(1, 3, 2)
ax[2] = plt.subplot(1, 3, 3, sharex=ax[0], sharey=ax[0])

ax[0].imshow(image, cmap=plt.cm.gray)
ax[0].set_title('Original')
ax[0].axis('off')

ax[1].imshow(binary, cmap=plt.cm.gray)
ax[1].set_title('Local')
ax[1].axis('off')

ax[2].imshow(binaryRefine, cmap=plt.cm.gray)
ax[2].set_title('Thresholded')
ax[2].axis('off')

plt.show()

In [ ]:
projections_average = remove_average_background_threshold(imagehaadf.data, sigma=30, thresh_variation = -3000)
projections_average_hs = hs.signals.Signal2D(projections_average[0:3], metadata = imagehaadf.metadata.as_dictionary())

In [ ]:
projections_average_iDPC = remove_average_background_mask(idpcsplit.data, binaryhs.data.astype(bool), distmasks)
projections_average_HAADF = remove_average_background_mask(haadfsplit.data, binaryhs.data.astype(bool), distmasks)
projections_average_iDPChs = hs.signals.Signal2D(projections_average_iDPC[0:3], metadata = idpcsplit.metadata.as_dictionary())
projections_average_HAADFhs = hs.signals.Signal2D(projections_average_HAADF[0:3], metadata = haadfsplit.metadata.as_dictionary())

In [ ]:
projections_average_iDPChs.save(r"iDPCdenoise_5-23d4CeO2BR.hspy")
projections_average_HAADFhs.save(r"HAADFdenoise_5-23d4CeO2BR.hspy")

## Option 2: Inpainting the mask

This is preferred - different inpainting options are offered from various Python packages. Multiple are demonstrated.

### HAADF and iDPC testing

If you are on a standard pc, run this section first. This rebins the images, performs inpainting with skimage (biharmonic) and cv2 (telea, based on fast marching) and generates a comparative plot.

skimage is generally a lot better, but comes at a computational cost. <br>
skimage biharmonic link: https://scikit-image.org/docs/0.21.x/api/skimage.restoration.html#skimage.restoration.inpaint_biharmonic

In [ ]:
idpc_test = sort_tilt2[2]
haadf_test = sort_tilt1[2]

masktest = mask[2]
imresize = resize(image.data, (256,256))
maskresize = resize(masktest, (256,256))
smoothed_mask = smooth_binary_mask(maskresize, sigma=10, threshold=0.2)

In [ ]:
image_defect =  imresize * ~smoothed_mask
image_result = inpaint.inpaint_biharmonic(image_defect, smoothed_mask, split_into_regions=True, channel_axis=None)#none is image greyscale

fig, axes = plt.subplots(ncols=2, nrows=2)
ax = axes.ravel()

ax[0].set_title('Original image')
ax[0].imshow(idpc_test)
ax[1].set_title('Mask')
ax[1].imshow(maskresize, cmap=plt.cm.gray)
ax[2].set_title('Defected image')
ax[2].imshow(image_defect)
ax[3].set_title('Inpainted image')
ax[3].imshow(image_result)

for a in ax:
    a.axis('off')

fig.tight_layout()
plt.show()

In [ ]:
int_mask = smoothed_mask.astype('uint8')

In [ ]:
image_defect =  imresize * ~maskresize
image_result = dst = cv2.inpaint(np.float32(image_defect),(maskresize.astype('uint8')),20,cv2.INPAINT_TELEA)

fig, axes = plt.subplots(ncols=2, nrows=2)
ax = axes.ravel()

ax[0].set_title('Original image')
ax[0].imshow(image_orig)
ax[1].set_title('Mask')
ax[1].imshow(maskresize, cmap=plt.cm.gray)
ax[2].set_title('Defected image')
ax[2].imshow(image_defect)
ax[3].set_title('Inpainted image')
ax[3].imshow(image_result)

for a in ax:
    a.axis('off')

fig.tight_layout()
plt.show()

### CV2 Inpaint

CV2 Telea inpainting. Faster, but inaccurate.

In [ ]:
inpaintRMidpc = []
inpaintidpc = []
inpaintRMhaadf = []
inpainthaadf = []

mask = binaryhs.data

for i in range(len(sort_tilt1)):
    image = sort_tilt1[i]
    image2 = sort_tilt2[i]
    masktest = mask[i]
    
    image_defect =  image.data * ~masktest 
    image_defect2 =  image2.data * ~masktest
    image_result = dst = cv2.inpaint(np.float32(image_defect),(masktest.astype('uint8')),100,cv2.INPAINT_TELEA)
    image_resulth = dst = cv2.inpaint(np.float32(image_defect2),(masktest.astype('uint8')),100,cv2.INPAINT_TELEA)
    
    inpaintidpc.append(image_result)
    inpainthaadf.append(image_resulth)   
    
    brresize = resize(image_result, (image.data.shape[0],image.data.shape[1]))
    brresize2 = resize(image_resulth, (image2.data.shape[0],image2.data.shape[1]))
    brresult = image.data - brresize
    brresult2 = image2.data - brresize2
    
    print(i)
    
    inpaintRMidpc.append(brresult)
    inpaintRMhaadf.append(brresult2)

### skimage inpaint

Skimage biharmonic inpainting (Laplacian). Accurate, but slow.

In [ ]:
inpaintRMidpc = []
inpaintidpc = []
inpaintRMhaadf = []
inpainthaadf = []

mask = binaryhs.data

for i in range(len(sort_tilt1)):
    image = sort_tilt1[i]
    image2 = sort_tilt2[i]
    masktest = mask[i]
    
    image_defect =  image.data * ~masktest
    image_defect2 =  image2.data * ~masktest
    image_result = dst = inpaint_biharmonic(np.float32(image_defect),(masktest.astype('uint8')))
    image_resulth = dst = inpaint_biharmonic(np.float32(image_defect2),(masktest.astype('uint8')))
    
    inpaintidpc.append(image_result)
    inpainthaadf.append(image_resulth)   
    
    brresize = resize(image_result, (image.data.shape[0],image.data.shape[1]))
    brresize2 = resize(image_resulth, (image2.data.shape[0],image2.data.shape[1]))
    brresult = image.data - brresize
    brresult2 = image2.data - brresize2

    # print statement for progress monitoring
    print(f"Image {i} completed")
    
    inpaintRMidpc.append(brresult)
    inpaintRMhaadf.append(brresult2)

### Continue from here!

In [ ]:
fig, axes = plt.subplots(ncols=2, nrows=2)
ax = axes.ravel()
i = 10

ax[0].imshow(inpaintidpc[i])
ax[1].imshow(inpaintRMidpc[i])
ax[2].imshow(inpainthaadf[i])
ax[3].imshow(inpaintRMhaadf[i])

for a in ax:
    a.axis('off')

fig.tight_layout()
plt.show()

In [ ]:
idpcstack = np.stack(inpaintRMidpc)
haadfstack = np.stack(inpaintRMhaadf)

brhaadf = hs.signals.Signal2D(haadfstack, metadata = imagehaadf.metadata.as_dictionary())
bridpc = hs.signals.Signal2D(idpcstack, metadata = image.metadata.as_dictionary())

In [ ]:
brhaadf.plot()

In [ ]:
bridpc.plot()

In [ ]:
bridpc.save(r"iDPC_CeO2_sorted_BR_inpaint.hspy")
brhaadf.save(r"HAADF_CeO2_sorted_BR_inpaint.hspy")

## Intensity normalisation

Intensity normalisation can help to reduce artefacts in the reconstruction.

In [ ]:
masksplit = mask
imagesplit = bridpc.split()
imagesplit2 = brhaadf.split()

In [ ]:
imagesplitref = imagesplit[13].data.copy()

In [ ]:
masktestnorm = masksplit[13].copy()
masktestnorm[masktestnorm>0.01] = 1
signal_mask = ~masktestnorm

min_signal_value = np.min(imagesplitref[signal_mask]) if np.any(signal_mask) else 0

image_corrected = np.copy(imagesplitref)
image_corrected = image_corrected - min_signal_value #(set min to 0)
image_corrected[~signal_mask] = 0 # set background also to 0

refsum = np.sum(image_corrected[signal_mask])
print(refsum)

In [ ]:
norm = []
shift = []
for i in range(len(imagesplit2)):
    image_shifted, image_normalised = normalise_to_reference_with_sum(imagesplit2[i].data.astype(np.float64), refsum, ~masksplit[i])
    print('normalised', i)
    normhs = hs.signals.Signal2D(image_normalised.astype(np.float32))
    # shifths is the image with the background shifted to 0
    shifths = hs.signals.Signal2D(image_shifted.astype(np.float32))
    norm.append(normhs)
    shift.append(shifths)

In [ ]:
normstack = hs.stack(norm)
imstack = hs.stack(shift)
normstack.plot()

In [ ]:
normstack.save(r"adfnorm_inpaintski_br.tif")